In [ ]:
# Full-Mamba HADA — PhysioNet 109-subject LOSO, AutoDIAL or source-only (one shard).
# Run 28 Kaggle jobs with JOB_INDEX=0..27; each job evaluates at most 4 target folds.
import codecs
import math
import os
import select
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ---------------- Job configuration ----------------
JOB_INDEX = 0          # Change to 0, 1, ..., 27 in separate Kaggle jobs.
FOLDS_PER_JOB = 4     # Sized to stay near the workload of the older 10-fold/175-epoch jobs.
MAX_EPOCHS = 75
TRAINING_MODE = 'autodial'  # Options: 'autodial', 'source_only'.
if TRAINING_MODE not in {'autodial', 'source_only'}:
    raise ValueError("TRAINING_MODE must be 'autodial' or 'source_only'")
USE_RA = False        # 64-channel Riemannian means are too costly for this full benchmark.
IM_TTA_STEPS = 0      # Keep the large-cohort run within the Kaggle time budget.

ALL_SUBJECTS = list(range(1, 110))
TOTAL_JOBS = math.ceil(len(ALL_SUBJECTS) / FOLDS_PER_JOB)
if not 0 <= JOB_INDEX < TOTAL_JOBS:
    raise ValueError(f'JOB_INDEX must be in 0..{TOTAL_JOBS - 1}')
start = JOB_INDEX * FOLDS_PER_JOB
TARGET_SUBJECTS = ALL_SUBJECTS[start:start + FOLDS_PER_JOB]
GPU_TARGETS = [TARGET_SUBJECTS[0::2], TARGET_SUBJECTS[1::2]]
JOB_TAG = f's{TARGET_SUBJECTS[0]:03d}_s{TARGET_SUBJECTS[-1]:03d}'

DATASET = 'physionet'
BRANCH = (
    'feature/hada-autodial-zhou-mi-preprocessing'
    if TRAINING_MODE == 'autodial'
    else 'feature/hada-full-mamba-no-da'
)
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
RUN_TAG = f'{DATASET}109_{TRAINING_MODE}_ep{MAX_EPOCHS}_{JOB_TAG}'
REPO_PATH = Path(f'/kaggle/working/tcformer-{RUN_TAG}')
MNE_DATA = Path('/kaggle/working/mne_data')
TRAIN_LOGS = [Path(f'/kaggle/working/{RUN_TAG}_gpu{gpu_id}.log') for gpu_id in range(2)]
RESULT_ARCHIVE = Path(f'/kaggle/working/{RUN_TAG}_results')


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return

    process = subprocess.Popen(
        command, cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
    )
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f'[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True)
            log_file.write(remaining)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Training exited with code {return_code}. Full log: {log_path}')


# ---------------- Clone the requested branch ----------------
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', '--force', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

# The base branch intentionally used only S001-S020. Expand the source pool to all 109
# subjects while keeping TARGET_SUBJECTS as the folds evaluated by this shard.
physionet_path = REPO_PATH / 'datamodules/physionet.py'
physionet_text = physionet_path.read_text(encoding='utf-8')
replacements = {
    'First-20-subject PhysioNet': 'Full-109-subject PhysioNet',
    'Each of the 19': 'Each of the 108',
    'all_subject_ids = list(range(1, 21))': 'all_subject_ids = list(range(1, 110))',
    'target subject must be in 1..20': 'target subject must be in 1..109',
    'Materializing all 20 subjects': 'Materializing all 109 subjects',
    'did not materialize subjects 1..20': 'did not materialize subjects 1..109',
    'if len(source_ids) != 19': 'if len(source_ids) != 108',
}
for old, new in replacements.items():
    if old not in physionet_text:
        raise RuntimeError(f'Expected PhysioNet source marker is missing: {old!r}')
    physionet_text = physionet_text.replace(old, new)
physionet_path.write_text(physionet_text, encoding='utf-8')
assert 'all_subject_ids = list(range(1, 110))' in physionet_text
assert 'if len(source_ids) != 108' in physionet_text

# Add a per-process subject override so two independent GPU workers can share
# this checkout without racing to rewrite the YAML configuration.
pipeline_path = REPO_PATH / 'train_pipeline.py'
pipeline_text = pipeline_path.read_text(encoding='utf-8')
gpu_arg_marker = '    parser.add_argument("--gpu_id", type=int, default=0, help="GPU device ID to use")\n'
subject_arg = (
    '    parser.add_argument("--subject_ids", type=int, nargs="+", default=None, ' 
    'help="Target subject IDs assigned to this process")\n'
)
if '--subject_ids' not in pipeline_text:
    if gpu_arg_marker not in pipeline_text:
        raise RuntimeError('Could not locate the gpu_id CLI marker in train_pipeline.py')
    pipeline_text = pipeline_text.replace(gpu_arg_marker, gpu_arg_marker + subject_arg, 1)
subject_override_marker = '    config["gpu_id"] = args.gpu_id\n'
subject_override = (
    '    if args.subject_ids is not None:\n'
    '        config["subject_ids"] = args.subject_ids\n'
)
if 'config["subject_ids"] = args.subject_ids' not in pipeline_text:
    if subject_override_marker not in pipeline_text:
        raise RuntimeError('Could not locate the gpu_id config marker in train_pipeline.py')
    pipeline_text = pipeline_text.replace(
        subject_override_marker, subject_override_marker + subject_override, 1
    )
pipeline_path.write_text(pipeline_text, encoding='utf-8')
model_text = (REPO_PATH / 'models/hada_tcformer.py').read_text(encoding='utf-8')
assert 'ModelCheckpoint' in pipeline_text and 'monitor="val_acc"' in pipeline_text
if TRAINING_MODE == 'autodial':
    assert 'class AutoDIALFeatureNorm' in model_text
else:
    assert 'ordinary labeled' in model_text and 'source batch' in model_text
print(f'Verified: 109-subject source pool, {TRAINING_MODE}, and source-validation checkpointing.')

# ---------------- Environment ----------------
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1',
     '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)

# ---------------- 75-epoch full-cohort override ----------------
if TRAINING_MODE == 'autodial':
    mode_override = (
        "c['domain_adaptation']=True; "
        "c['model_kwargs']['feature_alignment']='autodial'; "
    )
else:
    # Pure source-only: no target batches during training, no RA, and no IM-TTA.
    mode_override = (
        "c['domain_adaptation']=False; "
        "c['model_kwargs'].pop('feature_alignment', None); "
        "c['model_kwargs']['adversarial_weight']=0.0; "
        "c['model_kwargs']['mmd_weight']=0.0; "
        "c['model_kwargs']['temporal_mmd_weight']=0.0; "
    )
override = (
    "import yaml; from pathlib import Path; "
    "p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    f"c['max_epochs_loso']={MAX_EPOCHS}; "
    "c['log_every_n_epochs']=10; c['enable_progress_bar']=False; "
    "c['preprocessing']['physionet']['batch_size']=96; "
    "c['preprocessing']['physionet']['accumulate_grad_batches']=1; "
    "c['preprocessing']['physionet']['num_workers']=2; "
    f"c['preprocessing']['physionet']['riemannian_alignment']={USE_RA!r}; "
    f"c['preprocessing']['physionet'].setdefault('model_overrides', {{}})['im_tta_steps']={IM_TTA_STEPS}; "
    f"{mode_override}"
    "c['model_kwargs']['log_every_n_batches']=0; "
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); "
     "print('CUDA:', torch.cuda.is_available()); "
     "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; "
     "print('GPU count:', torch.cuda.device_count()); "
     "assert torch.cuda.device_count() >= 2, 'Select the Kaggle GPU T4 x2 accelerator'; "
     "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"])

# ---------------- Train / test this shard ----------------
MNE_DATA.mkdir(parents=True, exist_ok=True)
results_dir = REPO_PATH / 'results'
if results_dir.exists():
    shutil.rmtree(results_dir)
environment = os.environ.copy()
environment.update({
    'PYTHONUNBUFFERED': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'MPLBACKEND': 'Agg',
    'MNE_DATA': str(MNE_DATA),
    'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA),
})
print(f'===== {TRAINING_MODE.upper()} FULL-MAMBA HADA | PHYSIONET 109-SUBJECT LOSO | JOB {JOB_INDEX + 1}/{TOTAL_JOBS} ' 
      f'| TARGETS={TARGET_SUBJECTS} | GPU_SPLIT={GPU_TARGETS} | BATCH=96 | 75 EPOCHS | 4 CLASSES | 64 CH =====', flush=True)
if TRAINING_MODE == 'autodial':
    print('Protocol: each fold trains on 108 labeled source subjects; the held-out target is unlabeled during AutoDIAL training and used for final scoring only.', flush=True)
else:
    print('Protocol: each fold trains only on 108 labeled source subjects; target EEG and labels are excluded from training and adaptation, then used only for final scoring.', flush=True)
for gpu_id, subjects in enumerate(GPU_TARGETS):
    if subjects:
        print(f'GPU {gpu_id}: targets={subjects} | log={TRAIN_LOGS[gpu_id]}', flush=True)

def train_gpu(gpu_id, subjects):
    if not subjects:
        return
    command = [PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer',
               '--dataset', DATASET, '--loso', '--gpu_id', str(gpu_id),
               '--subject_ids', *map(str, subjects)]
    run(command, cwd=REPO_PATH, env=environment, stream=True,
        log_path=TRAIN_LOGS[gpu_id])

active_jobs = [(gpu_id, subjects) for gpu_id, subjects in enumerate(GPU_TARGETS) if subjects]
with ThreadPoolExecutor(max_workers=len(active_jobs)) as executor:
    futures = [executor.submit(train_gpu, gpu_id, subjects) for gpu_id, subjects in active_jobs]
    for future in futures:
        future.result()

if TRAINING_MODE == 'autodial':
    alpha_prefix = 'AutoDIAL alpha per feature group: '
    for gpu_id, subjects in active_jobs:
        log_text = TRAIN_LOGS[gpu_id].read_text(encoding='utf-8', errors='replace')
        alphas = [line.split(alpha_prefix, 1)[1].strip() for line in log_text.splitlines() if alpha_prefix in line]
        for subject_id, alpha in zip(subjects, alphas):
            print(f'S{subject_id:03d} final alpha (CNN groups 1-3, SSM group): {alpha}')

archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Completed targets:', TARGET_SUBJECTS)
print('Kaggle output archive:', archive)
print('Kaggle output logs:', [str(TRAIN_LOGS[gpu_id]) for gpu_id, _ in active_jobs])
